# Hotel Recommendation Model

**Author:** Zohaib Sheikh  
**Capstone:** TravelWise — Integrating MLOps in Travel Analytics  
**GitHub Repository:** https://github.com/zohaibsheikh007/Travel-MLOps-Capstone

## Project Summary

This notebook builds a collaborative-filtering hotel recommender for the TravelWise platform. We use truncated singular value decomposition (SVD) on the user-hotel rating matrix where the implicit signal is the total amount each user has spent at each hotel. The trained model is exposed through a Streamlit dashboard (`hotel.py`) which lets a sales agent pick a user and see the top-N suggested hotels.

## 1. Setup

In [ ]:
!pip install pandas numpy scipy scikit-learn matplotlib seaborn -q

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Load and inspect

In [ ]:
df = pd.read_csv('hotels.csv')
print(f'Rows: {len(df):,} | Columns: {df.shape[1]}')
df.head()

In [ ]:
print('Unique users :', df['userCode'].nunique())
print('Unique hotels:', df['name'].nunique())
print('Avg bookings per user:', df.groupby('userCode').size().mean().round(2))

## 3. EDA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df['name'].value_counts().head(10).plot(kind='barh', ax=ax[0], color='steelblue')
ax[0].set_title('Top 10 hotels by booking count')
df['place'].value_counts().head(10).plot(kind='barh', ax=ax[1], color='steelblue')
ax[1].set_title('Top 10 destinations')
plt.tight_layout(); plt.show()

## 4. Build the user-hotel interaction matrix

Filter to users with at least two stays so SVD can learn something. The rating is `total spend` per user-hotel pair (implicit feedback).

In [ ]:
active_users = df.groupby('userCode').size()
active_users = active_users[active_users >= 2].index
df_active = df[df['userCode'].isin(active_users)].copy()

le_hotel = LabelEncoder()
df_active['name_id'] = le_hotel.fit_transform(df_active['name'])
interactions = df_active.groupby(['userCode', 'name_id'])['price'].sum().reset_index()
print(f'Filtered interactions: {len(interactions):,}')

train, test = train_test_split(interactions, test_size=0.2, random_state=42)
pivot = train.pivot(index='userCode', columns='name_id', values='price').fillna(0)
matrix = pivot.values
user_index = list(pivot.index)
hotel_index = list(pivot.columns)
matrix.shape

## 5. Truncated SVD

In [ ]:
k = min(8, min(matrix.shape) - 1)
U, sigma, Vt = svds(matrix, k=k)
sigma = np.diag(sigma)
predictions = np.dot(np.dot(U, sigma), Vt)
preds_df = pd.DataFrame(predictions, columns=hotel_index, index=user_index)
print(f'Latent factor matrix: {predictions.shape}')

## 6. Evaluation

Reconstruction RMSE on held-out interactions. (Implicit-feedback recommenders are notoriously hard to score offline; we report this as a sanity check, not as the main user-facing metric.)

In [ ]:
actual, predicted = [], []
for _, row in test.iterrows():
    u, h = int(row['userCode']), int(row['name_id'])
    if u in preds_df.index and h in preds_df.columns:
        actual.append(row['price'])
        predicted.append(preds_df.loc[u, h])
rmse = np.sqrt(mean_squared_error(actual, predicted))
print(f'Held-out reconstruction RMSE: {rmse:.2f}')

## 7. Sample recommendations

In [ ]:
def recommend(user_id, top_n=5):
    if user_id not in preds_df.index:
        return f'User {user_id} not in training set.'
    scores = preds_df.loc[user_id].sort_values(ascending=False).head(top_n)
    out = pd.DataFrame({'hotel': le_hotel.inverse_transform(scores.index.astype(int)),
                        'score': scores.values.round(2)})
    return out

sample = preds_df.index[0]
print(f'Top 5 recommendations for user {sample}:')
recommend(sample)

## 8. Visualisation

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(predictions.flatten(), bins=60, color='steelblue')
plt.title('Distribution of latent affinity scores'); plt.tight_layout(); plt.show()

**Reasoning.** Matrix factorisation is a good fit here because the dataset is small (~40k bookings, ~1.3k users) and dense enough that explicit collaborative filtering works without needing implicit-feedback methods like ALS. The Streamlit app loads this same logic and lets the operator pick any user — the long-term plan is to swap SVD with LightFM once we have side-information (price tier, destination). That migration is explicitly called out as future work in the project report.